In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from mne.viz import plot_topomap

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from scipy.signal import spectrogram as sp_spectrogram  # noqa: E402
from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Combined-Features ICA on Wavelet Power

## Scope

This notebook performs ICA decomposition on wavelet power data where **all
three main dimensions — subjects, channels, and frequencies — are combined
into a single observation axis**, and **time** serves as the feature axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)  — 4-D wavelet power
Reshape: (n_subjects × n_channels × n_freqs,  n_times)
         ──────── observations ──────────────  features
```

Before reshaping the tensor is **z-scored along the time axis** so that
every `(subject, channel, frequency)` slice has zero mean and unit
variance.  This removes overall amplitude differences and ensures that
PCA/ICA operates on standardised activations.

## What the decomposition finds

PCA followed by ICA discovers a small set of **temporal component patterns**
(each of length T) shared across the S × C × F observations.  The ICA
score vector for each component can be reshaped back to
`(n_subjects, n_channels, n_freqs)` and decomposed into:

| Quantity | Shape | Interpretation |
|----------|-------|----------------|
| **Temporal component** | `(T,)` | A temporal pattern shared across observations |
| **Subject loadings** | `(S,)` | Mean absolute score — which participants contribute most |
| **Channel loadings** | `(C,)` | Spatial topography (scalp map) |
| **Frequency loadings** | `(F,)` | Spectral profile of the component |

## Analyses

1. Z-scoring and reshape
2. PCA dimensionality reduction + ICA decomposition
3. **(a)** Intersubject correlation matrix of ICA components
4. **(b)** Component loadings in time — mean ± standard deviation across subjects
5. **(c)** Frequency × Time mean-loading heatmaps of component activations
6. **(d)** Mean and variance of component loadings as scalp topomaps
7. **(e)** Per-subject loading bar plot for each component

## Configuration


In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ───────────────────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── Decomposition settings ────────────────────────────────────────────────────
N_COMPONENTS_PCA = 20  # number of PCA components to retain
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42  # reproducibility

# ── Storage directory ─────────────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "combined_features"
    / "pca_ica"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading


In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.


In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.


In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(
    f"Shape      : {bb_data.shape}  (subjects \u00d7 channels \u00d7 freqs \u00d7 times)"
)
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time series
to zero mean and unit variance.  This ensures that PCA/ICA are not
dominated by high-power channels, subjects, or frequency bands — every
observation contributes equally based on its temporal *pattern* rather
than its absolute amplitude.

**Reshaping** flattens the first three dimensions into a single
observation axis:

```
(S, C, F, T)  →  (S × C × F,  T)
                  observations  features
```

Each row of the resulting 2-D matrix is the z-scored power time course
of a single subject, at a single electrode, at a single wavelet
frequency.  PCA/ICA will discover shared temporal patterns across
these observations.


In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Reshape 4-D → 2-D:  (n_subjects * n_channels * n_freqs,  n_times)
X_z = bb_z.reshape(-1, n_times)  # (S*C*F, T)
print(f"Reshaped matrix shape  : {X_z.shape}")
print(f"  Observations (S×C×F) : {X_z.shape[0]}")
print(f"  Features     (T)     : {X_z.shape[1]}")
print(f"Row means  ≈ 0 : {X_z.mean(axis=1).mean():.6f}")
print(f"Row stds   ≈ 1 : {X_z.std(axis=1).mean():.6f}")

---
## Step 2 — PCA Dimensionality Reduction + ICA Decomposition

We first reduce the temporal feature space from T to `N_COMPONENTS_PCA`
principal components, keeping the directions of maximum variance.  Then
FastICA rotates the PCA subspace to maximise statistical independence,
yielding `N_COMPONENTS_ICA` independent components.

**Results:**

| Object | Shape | Description |
|--------|-------|-------------|
| `ica_scores` | `(S×C×F, K)` | Per-observation weight for each IC |
| `ica_components` | `(K, T)` | Temporal pattern of each IC |
| `scores_4d` | `(S, C, F, K)` | ICA scores reshaped back to the original axes |


In [ ]:
# --- PCA ---
pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
pca_scores = pca.fit_transform(X_z)  # (S*C*F, K_pca)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance explained")
axes[0].set_title(f"PCA Scree Plot \u2014 {LABEL}")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance \u2014 {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"Top {N_COMPONENTS_PCA} components explain "
    f"{cumulative[-1] * 100:.1f}% of total variance."
)

# --- ICA ---
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
ica_scores = ica.fit_transform(pca_scores)  # (S*C*F, K_ica)
ica_components = ica.components_ @ pca.components_  # (K_ica, T)

# Reshape ICA scores back to 4-D for downstream analysis
scores_4d = ica_scores.reshape(
    n_subjects, n_channels, n_freqs, N_COMPONENTS_ICA
)  # (S, C, F, K)

print(f"ICA scores shape       : {ica_scores.shape}")
print(f"ICA components shape   : {ica_components.shape}")
print(f"Scores 4-D shape       : {scores_4d.shape}")

---
## Analysis (a) — Intersubject Correlation Matrix of ICA Components

For each ICA component we compute a **subject × subject** Pearson
correlation matrix.  Each subject is represented by their
channel × frequency loading vector (shape `C × F`, flattened).  High
off-diagonal correlations indicate that the component has a consistent
spatial–spectral distribution across individuals — a hallmark of
stimulus-driven (rather than noise-driven) components.


In [ ]:
# Per-subject loading vector for each IC: flatten (C, F) → (C*F,)
# scores_4d: (S, C, F, K)  →  (S, C*F, K) after reshape
subject_cf_loadings = scores_4d.reshape(
    n_subjects, n_channels * n_freqs, N_COMPONENTS_ICA
)  # (S, C*F, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Each subject's (C*F,) loading vector for IC i
    corr_mat = np.corrcoef(subject_cf_loadings[:, :, i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Intersubject Correlation of IC Loadings \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "isc_component_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (b) — Component Loadings in Time (Mean ± Std Across Subjects)

The ICA decomposition produces **global** temporal component patterns
(shape `(K, T)`) shared across all observations.  To assess how
consistently each component is expressed across subjects, we compute
**per-subject temporal activations** as follows:

For subject *s* and component *k*:

$$a_{s,k}(t) = \frac{1}{C \cdot F}\sum_{c,f} \text{score}_{s,c,f,k}\;\cdot\;z_{s,c,f}(t)$$

where $z_{s,c,f}(t)$ is the z-scored wavelet power time series and
$\text{score}_{s,c,f,k}$ is the ICA score.  This gives a per-subject
weighted average of the original data projected through the component's
loading pattern.

We plot the **mean** across subjects (solid line) with a shaded band
showing ± 1 standard deviation, revealing when and how consistently
the component is active across individuals.


In [ ]:
# Per-subject temporal activations for each IC
# scores_4d: (S, C, F, K),  bb_z: (S, C, F, T)
# For each subject s, component k:
#   a_sk(t) = mean over (c, f) of [ score(s,c,f,k) * bb_z(s,c,f,t) ]
subject_temporal = np.einsum("scfk,scft->skt", scores_4d, bb_z) / (
    n_channels * n_freqs
)  # (S, K, T)

mean_temporal = subject_temporal.mean(axis=0)  # (K, T)
std_temporal = subject_temporal.std(axis=0)  # (K, T)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, mean_temporal[i], lw=0.8, color="darkorange", label="mean")
    ax.fill_between(
        time,
        mean_temporal[i] - std_temporal[i],
        mean_temporal[i] + std_temporal[i],
        alpha=0.25,
        color="darkorange",
        label="\u00b1 1 std",
    )
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"Component {i + 1} — Temporal Loading", fontsize=10)
    if i == 0:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"ICA Component Loadings in Time (mean \u00b1 std across subjects) \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_temporal_loadings.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (c) — Frequency × Time Mean-Loading Heatmap

For each ICA component, we compute the **mean loading per subject**
at each wavelet frequency and time point.  Instead of a secondary
STFT spectrogram, this directly leverages the existing wavelet
frequency decomposition:

```
loading(k, f, t) = mean_{s,c}[ scores_4d(s,c,f,k) × bb_z(s,c,f,t) ]
```

The heatmap shows which frequency bands and time intervals each IC
is most strongly associated with — bright regions indicate frequency–
time combinations where the IC loading is high on average across
subjects and channels.


In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
# Mean |loading| at each (freq, time): einsum over subjects and channels
ft_loading = np.einsum("scfk,scft->kft", scores_4d, bb_z) / (
    n_subjects * n_channels
)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_loading[i]  # (F, T)
    vmin_s, vmax_s = np.percentile(data_i, 1), np.percentile(data_i, 99)
    ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="inferno",
        vmin=vmin_s,
        vmax=vmax_s,
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} \u2014 Freq \u00d7 Time Mean Loading", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Frequency \u00d7 Time Mean Loading per IC \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_time_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (d) — Mean and Variance of Component Loadings as Topomaps

Channel loadings are obtained by averaging ICA scores over frequencies
for each subject, yielding a per-subject channel-loading matrix
`(S, C, K)`.  We then compute:

- **Mean** across subjects → `(C, K)` — the average spatial distribution
  of each IC.
- **Variance** across subjects → `(C, K)` — electrodes where the IC
  strength varies most between individuals.

Both are displayed as scalp topographic maps.


In [ ]:
# Per-subject channel loadings: average |scores| over frequencies → (S, C, K)
ica_channel_loadings = scores_4d.mean(axis=2)  # (S, C, K)

# Mean and variance across subjects
ica_ch_mean = ica_channel_loadings.mean(axis=0)  # (C, K)
ica_ch_var = ica_channel_loadings.var(axis=0)  # (C, K)

# Get MNE Info for topomap
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = min(6, N_COMPONENTS_ICA)

# --- Mean topomaps ---
_vlim_mean = np.percentile(np.abs(ica_ch_mean[:, :n_show]), 99)
fig_mean, axes_mean = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_mean = [axes_mean]

for i, ax in enumerate(axes_mean):
    im, _ = plot_topomap(
        ica_ch_mean[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        vlim=(-_vlim_mean, _vlim_mean),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_mean.suptitle(
    f"Mean Component Loading (topomap) \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_mean[-1], label="mean loading")
fig_mean.tight_layout()
if SAVE_PLOTS:
    fig_mean.savefig(PLOTS_DIR / "ica_topomap_mean.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

# --- Variance topomaps ---
_vmax_var = np.percentile(ica_ch_var[:, :n_show], 99)
fig_var, axes_var = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_var = [axes_var]

for i, ax in enumerate(axes_var):
    im, _ = plot_topomap(
        ica_ch_var[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        vlim=(0, _vmax_var),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_var.suptitle(
    f"Variance of Component Loading (topomap) \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_var[-1], label="variance")
fig_var.tight_layout()
if SAVE_PLOTS:
    fig_var.savefig(
        PLOTS_DIR / "ica_topomap_variance.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Analysis (e) — Per-Subject Loading Bar Plot for Each Component

For each ICA component, compute the **mean absolute score** across
channels and frequencies per subject.  This scalar summarises how
strongly each participant expresses the component.  Subjects with
uniformly high loadings indicate a stimulus-driven component;
uneven loadings may reflect individual differences (e.g. drug response).


In [ ]:
# Subject loadings: mean |score| over channels and frequencies
subject_loadings = np.abs(scores_4d).mean(axis=(1, 2))  # (S, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.barh(
        range(n_subjects),
        subject_loadings[:, i],
        color="darkorange",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_xlabel("|score|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Subject")
fig.suptitle(
    f"Per-Subject Loading per Component \u2014 {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_subject_loadings.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (f) — IC Temporal Patterns (Component Waveforms)

Each ICA component is a temporal pattern of shape `(T,)`. This plot shows the raw
waveform of each IC — the shared temporal structure that ICA extracted from the data.

**Interpretation:** Peaks and troughs indicate moments where the corresponding
subject–channel–frequency observations co-activate. Unlike analysis (b) which
shows per-subject *weighted* activations, this is the component waveform itself.

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.0 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, ica_components[i], lw=0.6, color="teal")
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"Component {i + 1} \u2014 Temporal Waveform", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"ICA Component Temporal Patterns \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_component_timecourse.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Analysis (g) — Per-Subject Per-Component Loading Heatmap

A 2-D heatmap with subjects on the y-axis and ICA components on the x-axis.
Each cell shows the mean |loading| for that subject–component pair, averaged
over channels and frequencies.

**Interpretation:** Rows that are uniformly bright indicate subjects whose data
strongly participates in all modes; columns that are bright indicate components
that are strongly expressed across all subjects.

In [ ]:
# Mean |loading| across channels and frequencies for each subject × IC
subject_loadings_hm = np.abs(scores_4d).mean(axis=(1, 2))  # (S, K)

fig, ax = plt.subplots(
    figsize=(max(8, N_COMPONENTS_ICA * 0.8), max(4, n_subjects * 0.4))
)
im = ax.imshow(subject_loadings_hm, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(N_COMPONENTS_ICA))
ax.set_xticklabels([f"IC {k + 1}" for k in range(N_COMPONENTS_ICA)], fontsize=9)
ax.set_yticks(range(n_subjects))
ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=9)
ax.set_xlabel("Component")
ax.set_ylabel("Subject")
ax.set_title(f"Per-Subject Per-Component Loading Heatmap \u2014 {LABEL}", fontsize=13)
plt.colorbar(im, ax=ax, label="mean |loading|")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_subject_component_heatmap.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (h) — Mean Loading in Frequency × Time (subject-averaged)

For each IC, compute the mean loading at every (frequency, time) cell, averaged
across subjects and channels: `loading(k, f, t) = mean_{s,c}[ score(s,c,f,k) × bb_z(s,c,f,t) ]`.

Y-axis = wavelet frequency, X-axis = time, colour = mean loading.
This directly leverages the wavelet decomposition's native frequency axis.

**Interpretation:** Bright regions show where the IC is most active at a given
frequency and time — revealing time–frequency specificity of each mode.

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
# Mean loading at each (freq, time): einsum over subjects and channels
ft_loading_h = np.einsum("scfk,scft->kft", scores_4d, bb_z) / (
    n_subjects * n_channels
)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_loading_h[i]  # (F, T)
    vmin_s, vmax_s = np.percentile(data_i, 1), np.percentile(data_i, 99)
    im = ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="inferno",
        vmin=vmin_s,
        vmax=vmax_s,
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} \u2014 Freq \u00d7 Time Mean Loading", fontsize=10)
    plt.colorbar(im, ax=ax, label="mean loading")

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Frequency \u00d7 Time Mean Loading per IC (subject-averaged) \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_freq_time_loading.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (i) — Subject-Consistency (Correlation) per Component

For each IC, compute the S×S intersubject correlation matrix (same as analysis a),
then extract the mean of the upper-triangle off-diagonal entries. This gives a
single consistency score per IC, plotted as a bar chart.

**Interpretation:** High bars indicate components that are consistently expressed
across subjects (shared neural response); low/negative bars indicate
subject-specific or noisy components.

In [ ]:
# Per-subject loading vector for each IC: flatten (C, F) → (C*F,)
subject_cf_load = scores_4d.reshape(
    n_subjects, n_channels * n_freqs, N_COMPONENTS_ICA
)  # (S, C*F, K)

isc_per_ic = np.zeros(N_COMPONENTS_ICA)
for k in range(N_COMPONENTS_ICA):
    corr_mat = np.corrcoef(subject_cf_load[:, :, k])  # (S, S)
    # Mean of upper-triangle off-diagonal entries
    triu_idx = np.triu_indices(n_subjects, k=1)
    isc_per_ic[k] = corr_mat[triu_idx].mean()

fig, ax = plt.subplots(figsize=(max(8, N_COMPONENTS_ICA * 0.7), 4))
colors = ["steelblue" if v >= 0 else "salmon" for v in isc_per_ic]
ax.bar(range(1, N_COMPONENTS_ICA + 1), isc_per_ic, color=colors)
ax.set_xlabel("Component")
ax.set_ylabel("Mean pairwise ISC (Pearson r)")
ax.set_xticks(range(1, N_COMPONENTS_ICA + 1))
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_title(f"Subject-Consistency per IC \u2014 {LABEL}", fontsize=13)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_subject_consistency_bar.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

# Print sorted consistency values
print("Subject-consistency (mean pairwise ISC) per IC:")
for k in np.argsort(isc_per_ic)[::-1]:
    print(f"  IC {k + 1}: {isc_per_ic[k]:.4f}")

---
## Analysis (j) — Frequency Profile per Component

For each IC, compute the mean |loading| within each canonical frequency band
(delta 1–4 Hz, theta 4–8, alpha 8–13, beta 13–30, gamma 30–70). This reveals
which frequency band dominates each component.

**Interpretation:** A component dominated by alpha means its spatial–temporal
pattern captures alpha-range wavelet power co-fluctuations. Components with
flat profiles span multiple bands equally.

In [ ]:
# Assign each wavelet frequency to its canonical band
band_names = list(FREQUENCY_BANDS.keys())
band_ranges = list(FREQUENCY_BANDS.values())
n_bands = len(band_names)

# Mean |loading| per band per IC: average |scores_4d| over subjects & channels,
# then group frequencies by band
freq_profile = np.abs(scores_4d).mean(axis=(0, 1))  # (F, K)

band_profile = np.zeros((n_bands, N_COMPONENTS_ICA))
for b_idx, (lo, hi) in enumerate(band_ranges):
    mask = (FREQS >= lo) & (FREQS < hi)
    if mask.sum() > 0:
        band_profile[b_idx] = freq_profile[mask].mean(axis=0)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

band_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
for i, ax in enumerate(axes):
    ax.barh(
        range(n_bands),
        band_profile[:, i],
        color=band_colors[:n_bands],
    )
    ax.set_yticks(range(n_bands))
    ax.set_yticklabels(band_names, fontsize=9)
    ax.set_xlabel("mean |loading|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Frequency band")
fig.suptitle(
    f"Frequency Profile per Component \u2014 {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_frequency_profile.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (k) — Spectrogram of Temporal Loadings (STFT, mean per participants)

Apply short-time Fourier transform (STFT) to the **subject-averaged temporal loading**
of each IC (`mean_temporal[k]`). This reveals the spectral content of each component's
temporal activation pattern and how it evolves over time.

**How this differs from analysis (h):** Analysis (h) uses the *wavelet* frequency axis
(the native decomposition frequencies). This analysis applies STFT to the IC's temporal
loading to see which *rhythmic patterns* (oscillation frequencies) are present in the
component's activation — independent of the wavelet bands.

In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
nperseg = min(256, n_times // 4)
noverlap = nperseg * 3 // 4

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    f_stft, t_stft, Sxx = sp_spectrogram(
        mean_temporal[i],
        fs=sfreq,
        nperseg=nperseg,
        noverlap=noverlap,
    )
    Sxx_db = 10 * np.log10(Sxx + 1e-12)
    vmin_s, vmax_s = np.percentile(Sxx_db, 1), np.percentile(Sxx_db, 99)
    ax.pcolormesh(
        t_stft,
        f_stft,
        Sxx_db,
        cmap="viridis",
        vmin=vmin_s,
        vmax=vmax_s,
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} \u2014 STFT Spectrogram of Temporal Loading", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Spectrogram of Mean Temporal Loadings (STFT) \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_stft_spectrogram.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Summary

### Variables available for further analysis

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X_z` | `(S×C×F, T)` | Z-scored reshaped 2-D matrix |
| `pca` | — | Fitted PCA object |
| `pca_scores` | `(S×C×F, K_pca)` | PCA-transformed scores |
| `ica` | — | Fitted FastICA object |
| `ica_scores` | `(S×C×F, K_ica)` | ICA scores (per-observation weights) |
| `ica_components` | `(K_ica, T)` | ICA temporal component patterns |
| `scores_4d` | `(S, C, F, K)` | ICA scores reshaped to 4-D |
| `subject_temporal` | `(S, K, T)` | Per-subject temporal activations |
| `ica_channel_loadings` | `(S, C, K)` | Per-subject channel loadings |

### Analyses implemented

| # | Analysis | Key finding |
|---|----------|-------------|
| (a) | Intersubject correlation matrix | Which ICs are consistent across participants |
| (b) | Temporal loadings (mean ± std) | When each IC is active and how variable across subjects |
| (c) | Freq × Time mean loading | Mean IC loading at each wavelet frequency and time |
| (d) | Mean / variance topomaps | Spatial distribution and inter-individual variability |
| (e) | Per-subject loading bars | Individual-level contribution to each IC |
| (f) | IC temporal patterns | Raw component waveforms — shared temporal structure |
| (g) | Subject × Component heatmap | Per-subject per-component loading strength |
| (h) | Freq × Time mean loading (subject-averaged) | Wavelet-domain frequency–time specificity |
| (i) | Subject-consistency bar plot | Mean pairwise ISC per component — one bar per IC |
| (j) | Frequency profile per component | Which canonical band dominates each IC |
| (k) | STFT spectrogram of loadings | Rhythmic content of IC temporal activations over time |

See `README.md` in this directory for the full analysis rationale,
alternative decomposition strategies, and ideas for future extensions.